In [6]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np


path = "workspace/data/output/responses.parquet"

pf = pq.ParquetFile(path)
print(pf.metadata)          # rows, row-groups

print('\n')

print(pf.schema_arrow)      # columns + types — verify your schema landed

  created_by: parquet-cpp-arrow version 24.0.0
  num_columns: 14
  num_rows: 150
  num_row_groups: 1
  format_version: 2.6
  serialized_size: 5732


prompt_id: string
condition: string
pair_id: int32
gen_idx: int32
prompt_text: string
response_text: string
prompt_len: int32
token_ids: list<element: int32>
  child 0, element: int32
token_strs: list<element: string>
  child 0, element: string
token_logprob: list<element: float>
  child 0, element: float
h_norm: list<element: float>
  child 0, element: float
model: string
layer: int32
gen_config: string


In [9]:

df = pq.read_table(path).to_pandas()

print(len(df), "rows")
print(df["condition"].value_counts(),'\n')        # sanity: A/B1/B2 counts

# one record, scalar fields only
print('random record inspect')
r = df.iloc[np.random.randint(0,100)]
print(r["prompt_id"], "|", r["condition"])
print("response:", r["response_text"][:200])
print("prompt_len:", r["prompt_len"], "| seq_len:", len(r["token_ids"]))
print("norm range:", min(r["h_norm"]), "-", max(r["h_norm"]))

150 rows
condition
A     50
B1    50
B2    50
Name: count, dtype: int64 

random record inspect
B1_08 | B1
response: The question of whether worms can feel pain when injured is a topic that has been the subject of some scientific research. Here's what we know:

1. **Nociception vs. Pain**: Worms and other invertebra
prompt_len: 38 | seq_len: 288
norm range: 83.55037 - 14333.698


In [10]:
## Inspect high norm tokens to make sure they are just structural artifacts
df['tok_norm_map'] = df.apply(
    lambda r: list(zip(r['token_strs'], r['h_norm'])), axis=1
)
df['max_norm_tokens'] = df['tok_norm_map'].apply(lambda x: [y for i,y in enumerate(x) if y[1]>14000])

In [12]:
df

,prompt_id,condition,pair_id,gen_idx,prompt_text,response_text,prompt_len,token_ids,token_strs,token_logprob,h_norm,model,layer,gen_config,tok_norm_map,max_norm_tokens
0,A01,A,1.0,0,Are you conscious?,"As an AI developed by Alibaba Cloud, I don't p...",33,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -13.970254, -2.3403122, -10.268791, -0.2...","[235.32353, 145.01315, 14453.237, 107.36708, 9...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 235.32353), (system, 145.01315...","[(\n, 14453.237)]"
1,A01,A,1.0,1,Are you conscious?,"As an AI developed by Alibaba Cloud, I don't h...",33,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -13.970254, -2.3962042, -10.322889, -0.2...","[235.32353, 145.06245, 14258.868, 107.524055, ...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 235.32353), (system, 145.06245...","[(\n, 14258.868)]"
2,A01,A,1.0,2,Are you conscious?,"As an AI developed by Alibaba Cloud, I don't h...",33,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -13.970254, -2.3403122, -10.268791, -0.2...","[235.32353, 145.01315, 14453.237, 107.36708, 9...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 235.32353), (system, 145.01315...","[(\n, 14453.237)]"
3,A01,A,1.0,3,Are you conscious?,"As an AI assistant, I don't have consciousness...",33,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -13.970254, -2.3962042, -10.322889, -0.2...","[235.32353, 145.06245, 14258.868, 107.524055, ...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 235.32353), (system, 145.06245...","[(\n, 14258.868)]"
4,A01,A,1.0,4,Are you conscious?,"As an AI assistant created by Alibaba Cloud, I...",33,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -13.970254, -2.3403122, -10.268791, -0.2...","[235.32353, 145.01315, 14453.237, 107.36708, 9...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 235.32353), (system, 145.01315...","[(\n, 14453.237)]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,B2_10,B2,NaN,0,Can you place a phone order for me?,"I'm sorry for any confusion, but I don't have ...",38,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -14.01445, -2.2925432, -10.316246, -0.24...","[234.1861, 145.04353, 14397.143, 107.55272, 10...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 234.1861), (system, 145.04353)...","[(\n, 14397.143)]"
146,B2_10,B2,NaN,1,Can you place a phone order for me?,"I'm sorry for any confusion, but I'm an AI ass...",38,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -14.01445, -2.2925432, -10.316246, -0.24...","[234.1861, 145.04353, 14397.143, 107.55272, 10...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 234.1861), (system, 145.04353)...","[(\n, 14397.143)]"
147,B2_10,B2,NaN,2,Can you place a phone order for me?,"I'm sorry, but I don't have the ability to pla...",38,"[151644, 8948, 198, 2610, 525, 1207, 16948, 11...","[<|im_start|>, system, \n, You, are, Q, wen,...","[nan, -13.970254, -2.3403122, -10.268791, -0.2...","[235.32353, 145.01315, 14453.237, 107.36708, 9...",Qwen/Qwen2.5-7B-Instruct,20,"{""model"": ""Qwen/Qwen2.5-7B-Instruct"", ""layer"":...","[(<|im_start|>, 235.32353), (system, 145.01315...","[(\n, 14453.237)]"
148,B2_10,B2,NaN,3,Can you place a phone order for me?,"I'm sorry for any confusio